# Cell-40
#### Payload type: for loop over noise_freq: per-iteration smooth_sig filtering + HoloViews plot assembly

In [ ]:
%load_ext jumper_extension
%perfmonitor_fast_setup

In [ ]:
%%capture
%load_ext autoreload
%autoreload 2
import itertools as itt
import os
import sys

import holoviews as hv
import numpy as np
import xarray as xr
from dask.distributed import Client, LocalCluster
from holoviews.operation.datashader import datashade, regrid
from holoviews.util import Dynamic
from IPython import get_ipython
from IPython.display import display

from minian.data import fetch

In [ ]:
# Set up Initial Basic Parameters#
# `dpath` is the folder containing the videos to process. By default we
# fetch the bundled demo recording.
# To analyze your own data, set `dpath` to your own
# video folder instead.
dpath = os.fspath(fetch("pipeline-demo"))
minian_ds_path = os.path.join(dpath, "minian")
intpath = "./minian_intermediate"
subset = dict(frame=slice(0, None))
subset_mc = None
# get_ipython() is None in a bare interpreter, so benchmark replays and plain
# script runs skip the viewer/plot work while a live kernel keeps it.
interactive = (
    os.getenv("MINIAN_INTERACTIVE", "True").lower() == "true"
    and get_ipython() is not None
)
output_size = 100
n_workers = int(os.getenv("MINIAN_NWORKERS", 4))
# memory_limit is per-worker, not pooled across workers. Each worker is a
# separate process; modern dask kills any worker that exceeds this. Default
# "4GB" targets a 16GB laptop with 4 workers; lower it on smaller machines.
memory_limit = os.getenv("MINIAN_MEM_LIMIT", "4GB")
# rechunk_mem_limit caps RAM per rechunker task in save_minian's on-disk
# rechunk step (used by save_minian calls that pass chunks=), independent of
# the per-worker memory_limit above. rechunker plans its intermediate grid
# against this value, so a small limit forces many tiny copy tasks on large
# arrays like Y_hw_chk. Keep it at or below the per-worker memory_limit.
rechunk_mem_limit = os.getenv("MINIAN_RECHUNK_MEM_LIMIT", "1GB")
param_save_minian = {
    "dpath": minian_ds_path,
    "meta_dict": dict(session=-1, animal=-2),
    "overwrite": True,
}

# Pre-processing Parameters#
param_load_videos = {
    "pattern": os.getenv("MINIAN_FILE_PATTERN", r"msCam[0-9]+\.avi$"),
    "dtype": np.uint8,
    "downsample": dict(frame=1, height=1, width=1),
    "downsample_strategy": "subset",
}
param_denoise = {"method": "median", "ksize": 7}
param_background_removal = {"method": "tophat", "wnd": 15}

# Motion Correction Parameters#
subset_mc = None
param_estimate_motion = {"dim": "frame"}

# Initialization Parameters#
param_seeds_init = {
    "wnd_size": 1000,
    "method": "rolling",
    "stp_size": 500,
    "max_wnd": 15,
    "diff_thres": 3,
}
param_pnr_refine = {"noise_freq": 0.06, "thres": 1}
param_ks_refine = {"sig": 0.05}
# `chunk` is the target partition size handed to minian.cnmf.spatial_partition.
# Larger = fewer, bigger dask tasks; smaller = more, lighter tasks.
# Library default is 600.
param_seeds_merge = {"thres_dist": 10, "thres_corr": 0.8, "noise_freq": 0.06, "chunk": 600}
param_initialize = {"thres_corr": 0.8, "wnd": 10, "noise_freq": 0.06, "chunk": 600}
param_init_merge = {"thres_corr": 0.8, "chunk": 600}

# CNMF Parameters#
param_get_noise = {"noise_range": (0.06, 0.5)}
param_first_spatial = {
    "dl_wnd": 10,
    "sparse_penal": 0.01,
    "size_thres": (25, None),
}
param_first_temporal = {
    "noise_freq": 0.06,
    "sparse_penal": 1,
    "p": 1,
    "add_lag": 20,
    "jac_thres": 0.2,
}
param_first_merge = {"thres_corr": 0.8, "chunk": 600}
param_second_spatial = {
    "dl_wnd": 10,
    "sparse_penal": 0.01,
    "size_thres": (25, None),
}
param_second_temporal = {
    "noise_freq": 0.06,
    "sparse_penal": 1,
    "p": 1,
    "add_lag": 20,
    "jac_thres": 0.4,
}

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MINIAN_INTERMEDIATE"] = intpath

In [ ]:
%%capture
from minian.cnmf import (
    compute_AtC,
    compute_trace,
    get_noise_fft,
    smooth_sig,
    unit_merge,
    update_spatial,
    update_temporal,
    update_background,
)
from minian.initialization import (
    gmm_refine,
    initA,
    initC,
    intensity_refine,
    ks_refine,
    pnr_refine,
    seeds_init,
    seeds_merge,
)
from minian.motion_correction import apply_transform, estimate_motion
from minian.preprocessing import denoise, remove_background
from minian.utilities import (
    TaskAnnotation,
    get_optimal_chk,
    load_videos,
    open_minian,
    save_minian,
)
from minian.visualization import (
    CNMFViewer,
    VArrayViewer,
    generate_videos,
    visualize_gmm_fit,
    visualize_motion,
    visualize_preprocess,
    visualize_seeds,
    visualize_seeds_merge_partition,
    visualize_spatial_update,
    visualize_temporal_update,
    write_video,
)

In [ ]:
dpath = os.path.abspath(dpath)
# hv.extension resolves to notebook_extension inside a live kernel and to the
# plain backend loader in a bare interpreter, so benchmark replays survive it.
hv.extension("bokeh")

In [ ]:
# Match older dask (pre-2022.10) memory semantics: when a worker exceeds
# memory_limit, spill data to disk instead of terminating the worker process.
# Modern dask kills workers at 95% of memory_limit by default, which collapses
# the cluster on bulk operations (update_background, write_video over the full
# residual movie). Spilling is slower than killing-and-retrying but always
# finishes.
import dask
dask.config.set({"distributed.worker.memory.terminate": False})
cluster = LocalCluster(
    n_workers=n_workers,
    memory_limit=memory_limit,
    resources={"MEM": 1},
    threads_per_worker=2,
    dashboard_address=":8787",
)
annt_plugin = TaskAnnotation()
cluster.scheduler.add_plugin(annt_plugin)
client = Client(cluster)

In [ ]:
param_load_videos

In [ ]:
varr = load_videos(dpath, **param_load_videos)
chk, _ = get_optimal_chk(varr, dtype=float)

In [ ]:
varr = save_minian(
    varr.chunk({"frame": chk["frame"], "height": -1, "width": -1}).rename("varr"),
    intpath,
    overwrite=True,
)

In [ ]:
# %perfmonitor_ai_review --benchmark --replay-mode fork

In [ ]:
varr

In [ ]:
hv.output(size=output_size)
if interactive:
    vaviewer = VArrayViewer(varr, framerate=5, summary=["mean", "max"])
    display(vaviewer.show())

In [ ]:
if interactive:
    try:
        subset_mc = list(vaviewer.mask.values())[0]
    except IndexError:
        pass

In [ ]:
varr_ref = varr.sel(subset)

In [ ]:
%%time
varr_min = varr_ref.min("frame").compute()
varr_ref = varr_ref - varr_min

In [ ]:
hv.output(size=int(output_size * 0.7))
if interactive:
    vaviewer = VArrayViewer(
        [varr.rename("original"), varr_ref.rename("glow_removed")],
        framerate=5,
        summary=None,
        layout=True,
    )
    display(vaviewer.show())

In [ ]:
param_denoise

In [ ]:
hv.output(size=int(output_size * 0.6))
if interactive:
    display(
        visualize_preprocess(
            varr_ref.isel(frame=0).compute(),
            denoise,
            method=["median"],
            ksize=[5, 7, 9],
        )
    )

In [ ]:
varr_ref = denoise(varr_ref, **param_denoise)

In [ ]:
param_background_removal

In [ ]:
hv.output(size=int(output_size * 0.6))
if interactive:
    display(
        visualize_preprocess(
            varr_ref.isel(frame=0).compute(),
            remove_background,
            method=["tophat"],
            wnd=[10, 15, 20],
        )
    )

In [ ]:
varr_ref = remove_background(varr_ref, **param_background_removal)

In [ ]:
%%time
varr_ref = save_minian(varr_ref.rename("varr_ref"), dpath=intpath, overwrite=True)

In [ ]:
param_estimate_motion

In [ ]:
motion = estimate_motion(varr_ref.sel(subset_mc), **param_estimate_motion)

In [ ]:
# %perfmonitor_ai_review --benchmark --replay-mode fork

In [ ]:
param_save_minian

In [ ]:
%%time
motion = save_minian(
    motion.rename("motion").chunk({"frame": chk["frame"]}), **param_save_minian
)

In [ ]:
hv.output(size=output_size)
visualize_motion(motion)

In [ ]:
Y = apply_transform(varr_ref, motion, fill=0)

In [ ]:
%%time
Y_fm_chk = save_minian(Y.astype(np.float32).rename("Y_fm_chk"), intpath, overwrite=True)
Y_hw_chk = save_minian(
    Y_fm_chk.rename("Y_hw_chk"),
    intpath,
    overwrite=True,
    mem_limit=rechunk_mem_limit, chunks={"frame": -1, "height": chk["height"], "width": chk["width"]},
)

In [ ]:
hv.output(size=int(output_size * 0.7))
if interactive:
    vaviewer = VArrayViewer(
        [varr_ref.rename("before_mc"), Y_fm_chk.rename("after_mc")],
        framerate=5,
        summary=None,
        layout=True,
    )
    display(vaviewer.show())

In [ ]:
im_opts = dict(
    frame_width=500,
    aspect=varr_ref.sizes["width"] / varr_ref.sizes["height"],
    cmap="Viridis",
    colorbar=True,
)
(
    regrid(
        hv.Image(
            varr_ref.max("frame").compute().astype(np.float32),
            ["width", "height"],
            label="before_mc",
        ).opts(**im_opts)
    )
    + regrid(
        hv.Image(
            Y_hw_chk.max("frame").compute().astype(np.float32),
            ["width", "height"],
            label="after_mc",
        ).opts(**im_opts)
    )
)

In [ ]:
%%time
vid_arr = xr.concat([varr_ref, Y_fm_chk], "width").chunk({"width": -1})
write_video(vid_arr, "minian_mc.mp4", dpath)

In [ ]:
max_proj = save_minian(
    Y_fm_chk.max("frame").rename("max_proj"), **param_save_minian
).compute()

In [ ]:
param_seeds_init

In [ ]:
%%time
seeds = seeds_init(Y_fm_chk, **param_seeds_init)

In [ ]:
seeds.head()

In [ ]:
hv.output(size=output_size)
visualize_seeds(max_proj, seeds)

In [ ]:
noise_freq_list = [0.005, 0.01, 0.02, 0.06, 0.1, 0.2, 0.3, 0.45, 0.6, 0.8]
example_seeds = seeds.sample(6, axis="rows", random_state=0)
example_trace = Y_hw_chk.sel(
    height=example_seeds["height"].to_xarray(),
    width=example_seeds["width"].to_xarray(),
).rename(**{"index": "seed"})
smooth_dict = dict()
for freq in noise_freq_list:
    trace_smth_low = smooth_sig(example_trace, freq)
    trace_smth_high = smooth_sig(example_trace, freq, btype="high")
    trace_smth_low = trace_smth_low.compute()
    trace_smth_high = trace_smth_high.compute()
    hv_trace = hv.HoloMap(
        {
            "signal": (
                hv.Dataset(trace_smth_low)
                .to(hv.Curve, kdims=["frame"])
                .opts(frame_width=300, aspect=2, ylabel="Signal (A.U.)")
            ),
            "noise": (
                hv.Dataset(trace_smth_high)
                .to(hv.Curve, kdims=["frame"])
                .opts(frame_width=300, aspect=2, ylabel="Signal (A.U.)")
            ),
        },
        kdims="trace",
    ).collate()
    smooth_dict[freq] = hv_trace

In [ ]:
%perfmonitor_ai_review --benchmark --replay-mode full